# Flickr30k T2→S2 Sequence KD — seed 123

Confirmation run đã được đăng ký trước cùng seed 42 và không thay đổi phương
pháp sau khi xem kết quả. Loss:
`L = 0,5 CE_human + 0,5 CE_teacher_sequence`.

## Kaggle input bắt buộc — đúng 6 input

1. Output `F30K_Data`.
2. Output `F30K_Base`.
3. Output `F30K_S2_123`.
4. Output `F30K_T2_Pseudo_Full`.
5. Dataset `Flickr Image dataset`.
6. Package/dataset `pycocoevalcap-1.2` offline.

Khi resume, gắn thêm đúng một output gần nhất của `F30K_T2S2_SeqKD_123`.
Không gắn output KD seed khác và không gắn Teacher model.

Runtime: GPU T4, Internet Off, dự kiến 2–4 giờ. Tối đa 20 epoch; Early
Stopping từ epoch 5, patience 4; Test `SEALED / NOT USED`.

## Cell 1 — Authorization, integrity và Test-seal

In [1]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, subprocess, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MODEL_ID = "S2"
MODEL_NAME = "MobileViT-S"
SEED = 123
EXPERIMENT_ID = "F30K_T2S2_SeqKD_123"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def unique_named(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (f"Cần đúng một {name}", matches)
    return matches[0]

BASE_READY_PATH = unique_named("F30K_BASE_READY.json")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
PROTOCOL_PATH = unique_named("F30K_BASELINE_PROTOCOL.json")
TRAIN_PATH = unique_named("f30k_train.parquet")
VAL_PATH = unique_named("f30k_validation.parquet")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")

base_ready = json.loads(BASE_READY_PATH.read_text(encoding="utf-8"))
data_contract = json.loads(DATA_CONTRACT_PATH.read_text(encoding="utf-8"))
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))

assert base_ready["status"] == "authorized"
assert base_ready["test_set_status"] == "sealed"
assert base_ready["test_set_used"] is False
assert (
    SEED == int(protocol["development_seed"])
    or SEED in [int(x) for x in protocol["confirmation_seeds"]]
)
assert protocol["vocabulary_size"] == 7456 == len(vocab["tokens"])
assert protocol["maximum_epochs"] == 20
assert protocol["selection_metric"] == "Validation CIDEr-D"
assert data_contract["test_set_status"] == "sealed"
assert data_contract["test_caption_exported"] is False

for file_name, expected in {
    "f30k_train.parquet": data_contract["artifacts"]["train"]["sha256"],
    "f30k_validation.parquet": data_contract["artifacts"]["validation"]["sha256"],
    "f30k_vocabulary_minfreq5.json": data_contract["artifacts"]["vocabulary"]["sha256"],
}.items():
    actual_path = {"f30k_train.parquet": TRAIN_PATH,
                   "f30k_validation.parquet": VAL_PATH,
                   "f30k_vocabulary_minfreq5.json": VOCAB_PATH}[file_name]
    assert sha256(actual_path) == expected, file_name

if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    if wheels:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])])
    else:
        packages = [p for p in INPUT_ROOT.rglob("pycocoevalcap") if p.is_dir()]
        assert packages, "Hãy gắn input pycocoevalcap-1.2 offline."
        sys.path.insert(0, str(packages[0].parent))

assert importlib.util.find_spec("pycocoevalcap") is not None
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."
assert torch.cuda.is_available(), "Notebook này yêu cầu GPU T4."
DEVICE = torch.device("cuda")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("Preflight             : PASSED")
print("Experiment            :", EXPERIMENT_ID)
print("Model                 :", MODEL_NAME)
print("Train/Validation      : VERIFIED")
print("Vocabulary            :", len(vocab["tokens"]))
print("Selection             : Validation CIDEr-D")
print("Test                   : SEALED / NOT READ")

Preflight             : PASSED
Experiment            : F30K_T2S2_SeqKD_123
Model                 : MobileViT-S
Train/Validation      : VERIFIED
Vocabulary            : 7456
Selection             : Validation CIDEr-D
Test                   : SEALED / NOT READ


## Cell 2 — Student architecture và Flickr30k loaders

In [2]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
assert train_df["image_id"].nunique() == 29000 and len(train_df) == 145000
assert val_df["image_id"].nunique() == 1014 and len(val_df) == 5070
assert set(train_df.image_id).isdisjoint(set(val_df.image_id))

needed_names = {f"{int(x)}.jpg" for x in pd.concat([train_df.image_id, val_df.image_id])}
image_candidates = [p for p in INPUT_ROOT.rglob("*.jpg") if p.name in needed_names]
IMAGE_BY_NAME = {}
duplicate_alias_count = 0
for path in image_candidates:
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
        continue
    if old.resolve() == path.resolve():
        continue
    # Một số bản đóng gói Kaggle lưu cùng ảnh ở cả
    # flickr30k_images/<file> và flickr30k_images/flickr30k_images/<file>.
    # Chấp nhận alias lồng nhau khi kích thước byte giống hệt và luôn chọn
    # đường dẫn nông/ngắn nhất. Nếu kích thước khác nhau, dừng để tránh ánh xạ
    # nhầm hai ảnh chỉ tình cờ có cùng tên.
    assert old.stat().st_size == path.stat().st_size, (
        "Hai ảnh cùng tên nhưng khác kích thước", path.name, old, path
    )
    duplicate_alias_count += 1
    IMAGE_BY_NAME[path.name] = min(
        [old, path],
        key=lambda p: (len(p.parts), len(str(p)), str(p)),
    )
missing = sorted(needed_names - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Flickr30k", len(missing), missing[:10])

train_groups = []
for image_id, group in train_df.groupby("image_id", sort=True):
    train_groups.append((int(image_id), [list(map(int, x)) for x in group.token_ids]))
val_groups = []
VAL_REFERENCES = {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    val_groups.append((iid, [list(map(int, x)) for x in group.token_ids]))
    VAL_REFERENCES[iid] = [str(x) for x in group.caption.tolist()]

MEAN = [0.0, 0.0, 0.0]
STD = [1.0, 1.0, 1.0]
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CaptionDataset(Dataset):
    def __init__(self, rows, transform, training):
        self.rows, self.transform, self.training, self.epoch = rows, transform, training, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        caption = captions[(self.epoch + image_id % len(captions)) % len(captions)] if self.training else captions[0]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as im:
            image = self.transform(im.convert("RGB"))
        return image_id, image, torch.tensor(caption, dtype=torch.long)

def collate(batch):
    image_ids, images, captions = zip(*batch)
    max_len = max(len(x) for x in captions)
    tokens = torch.zeros(len(captions), max_len, dtype=torch.long)
    mask = torch.zeros(len(captions), max_len, dtype=torch.bool)
    for i, x in enumerate(captions):
        tokens[i, :len(x)] = x
        mask[i, :len(x)] = True
    return torch.tensor(image_ids), torch.stack(images), tokens, mask

train_set = CaptionDataset(train_groups, train_transform, True)
val_set = CaptionDataset(val_groups, val_transform, False)

def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    g = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(train_set, batch_size=32, shuffle=True, generator=g, num_workers=2,
                      pin_memory=True, persistent_workers=False, collate_fn=collate)

val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=True, collate_fn=collate)

def select_tensor(x):
    if torch.is_tensor(x):
        return x
    if isinstance(x, dict):
        return [v for v in x.values() if torch.is_tensor(v)][-1]
    if isinstance(x, (tuple, list)):
        return [v for v in x if torch.is_tensor(v)][-1]
    raise TypeError(type(x))

class VisualEncoder(nn.Module):
    def __init__(self, backbone, source_dim):
        super().__init__()
        self.backbone = backbone
        self.source_dim = source_dim
        self.projection = nn.Linear(source_dim, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)
        nn.init.xavier_uniform_(self.projection.weight)
        nn.init.zeros_(self.projection.bias)
        nn.init.normal_(self.visual_position_embedding, std=0.02)
    def forward(self, images):
        if MODEL_ID == "S1":
            feat = self.backbone(images)
        else:
            feat = select_tensor(self.backbone.forward_features(images))
        assert feat.ndim == 4 and feat.shape[1] == self.source_dim and tuple(feat.shape[-2:]) == (7, 7), feat.shape
        tokens = feat.flatten(2).transpose(1, 2).contiguous()
        return self.output_norm(self.projection(tokens) + self.visual_position_embedding)

class CaptionDecoder(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.d_model, self.max_steps = 256, 31
        self.token_embedding = nn.Embedding(vocabulary_size, 256, padding_idx=0)
        self.position_embedding = nn.Embedding(31, 256)
        self.embedding_dropout = nn.Dropout(0.1)
        layer = nn.TransformerDecoderLayer(256, 8, 1024, 0.1, "gelu",
                                           batch_first=True, norm_first=True)
        self.transformer_decoder = nn.TransformerDecoder(layer, 3)
        self.final_norm = nn.LayerNorm(256)
        self.output_bias = nn.Parameter(torch.zeros(vocabulary_size))
        nn.init.normal_(self.token_embedding.weight, std=256 ** -0.5)
        nn.init.normal_(self.position_embedding.weight, std=0.02)
        with torch.no_grad():
            self.token_embedding.weight[0].zero_()
    def forward(self, token_ids, memory, attention_mask=None):
        n, t = token_ids.shape
        pos = torch.arange(t, device=token_ids.device).unsqueeze(0).expand(n, -1)
        x = self.embedding_dropout(self.token_embedding(token_ids) * 16.0 + self.position_embedding(pos))
        causal = torch.triu(torch.ones(t, t, dtype=torch.bool, device=token_ids.device), diagonal=1)
        key_pad = None if attention_mask is None else ~attention_mask.bool()
        x = self.transformer_decoder(x, memory, tgt_mask=causal, tgt_key_padding_mask=key_pad)
        return F.linear(self.final_norm(x), self.token_embedding.weight, self.output_bias)

if MODEL_ID == "S1":
    from torchvision.models import mobilenet_v3_large
    base = mobilenet_v3_large(weights=None).features
elif MODEL_ID == "S2":
    import timm
    base = timm.create_model("mobilevit_s.cvnets_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
elif MODEL_ID == "S3":
    import timm
    base = timm.create_model("tiny_vit_5m_224.dist_in22k_ft_in1k", pretrained=False)
    base.reset_classifier(num_classes=0, global_pool="")
    for head_name in ["head", "head_dist", "classifier", "fc"]:
        head = getattr(base, head_name, None)
        if isinstance(head, nn.Module) and not isinstance(head, nn.Identity):
            setattr(base, head_name, nn.Identity())
else:
    raise ValueError(MODEL_ID)

encoder = VisualEncoder(base, 640)

# Decoder Flickr30k được khởi tạo mới bằng SEED hiện tại; không nạp decoder
# MS COCO có vocabulary 10.080.
torch.manual_seed(SEED)
decoder = CaptionDecoder(len(vocab["tokens"]))
encoder.to(DEVICE)
decoder.to(DEVICE)

for module in encoder.modules():
    if isinstance(module, nn.modules.batchnorm._BatchNorm):
        module.eval()

enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
assert enc_params == 5114784, (enc_params, 5114784)
assert dec_params == 5_084_960, dec_params
print("Images train/val       :", len(train_groups), "/", len(val_groups))
print("Nested image aliases   :", duplicate_alias_count, "(deduplicated)")
print("Encoder/decoder params :", enc_params, "/", dec_params)
print("Test                   : SEALED / NOT READ")

Images train/val       : 29000 / 1014
Nested image aliases   : 30014 (deduplicated)
Encoder/decoder params : 5114784 / 5084960
Test                   : SEALED / NOT READ


## Cell 3 — Baseline initialization và Sequence-KD pairing

In [3]:
TEACHER_ID = "T2"
TEACHER_NAME = "BLIP Base"
BASELINE_EXPERIMENT_ID = "F30K_S2_123"
BASELINE_RESULT_NAME = BASELINE_EXPERIMENT_ID + "_RESULT.json"
PSEUDO_EXPERIMENT_ID = "F30K_T2_Pseudo_Full"
PSEUDO_RESULT_NAME = PSEUDO_EXPERIMENT_ID + "_RESULT.json"

BASELINE_RESULT_PATH = unique_named(BASELINE_RESULT_NAME)
PSEUDO_RESULT_PATH = unique_named(PSEUDO_RESULT_NAME)
PSEUDO_PARQUET_PATH = unique_named("pseudocaptions.parquet")
baseline_result = json.loads(BASELINE_RESULT_PATH.read_text(encoding="utf-8"))
pseudo_result = json.loads(PSEUDO_RESULT_PATH.read_text(encoding="utf-8"))

assert baseline_result["status"] == "complete"
assert baseline_result["experiment_id"] == BASELINE_EXPERIMENT_ID
assert baseline_result["model_id"] == MODEL_ID
assert int(baseline_result["seed"]) == SEED == 123
assert baseline_result["test_set_status"] == "sealed"
assert baseline_result["test_set_used"] is False
assert pseudo_result["status"] == "complete"
assert pseudo_result["experiment_id"] == PSEUDO_EXPERIMENT_ID
assert pseudo_result["teacher_id"] == TEACHER_ID
assert pseudo_result["matched_student"].startswith(MODEL_ID)
assert pseudo_result["method"] == "Sequence KD"
assert pseudo_result["records"] == pseudo_result["unique_image_ids"] == 29000
assert pseudo_result["sequence_kd_ready"] is True
assert pseudo_result["next_phase_authorized"] == "F30K_T2S2_SeqKD_42"
assert SEED in (42, 123, 2026)
assert pseudo_result["test_set_status"] == "sealed"
assert pseudo_result["test_set_used"] is False
assert sha256(PSEUDO_PARQUET_PATH) == pseudo_result["parquet"]["sha256"]

BASELINE_BEST_PATH = BASELINE_RESULT_PATH.parent / "best.pt"
assert BASELINE_BEST_PATH.is_file()
assert BASELINE_BEST_PATH.stat().st_size == baseline_result["artifacts"]["best.pt"]["bytes"]
assert sha256(BASELINE_BEST_PATH) == baseline_result["artifacts"]["best.pt"]["sha256"]

pseudo_df = pd.read_parquet(PSEUDO_PARQUET_PATH)
assert len(pseudo_df) == pseudo_df["image_id"].nunique() == 29000
assert set(pseudo_df["image_id"].astype(int)) == set(train_df["image_id"].astype(int))
PSEUDO_BY_ID = {
    int(row.image_id): list(map(int, row.student_token_ids))
    for row in pseudo_df.itertuples(index=False)
}
assert all(values[0] == 1 and values[-1] == 2 for values in PSEUDO_BY_ID.values())
assert all(all(0 <= token < len(vocab["tokens"]) for token in values)
           for values in PSEUDO_BY_ID.values())

baseline_payload = torch.load(BASELINE_BEST_PATH, map_location="cpu", weights_only=False)
assert baseline_payload["experiment_id"] == BASELINE_EXPERIMENT_ID
assert baseline_payload["seed"] == SEED
assert baseline_payload["test_set_used"] is False
encoder.load_state_dict(baseline_payload["encoder_state_dict"], strict=True)
decoder.load_state_dict(baseline_payload["decoder_state_dict"], strict=True)
del baseline_payload

class SequenceKDDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform, self.epoch = rows, transform, 0
    def set_epoch(self, epoch):
        self.epoch = int(epoch)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        image_id, captions = self.rows[index]
        human = captions[(self.epoch + image_id % len(captions)) % len(captions)]
        pseudo = PSEUDO_BY_ID[int(image_id)]
        with Image.open(IMAGE_BY_NAME[f"{image_id}.jpg"]) as image:
            pixels = self.transform(image.convert("RGB"))
        return (
            int(image_id), pixels,
            torch.tensor(human, dtype=torch.long),
            torch.tensor(pseudo, dtype=torch.long),
        )

def sequence_collate(rows):
    image_ids, images, human, pseudo = zip(*rows)
    def pad(values):
        maximum = max(len(x) for x in values)
        tokens = torch.zeros(len(values), maximum, dtype=torch.long)
        mask = torch.zeros(len(values), maximum, dtype=torch.bool)
        for index, item in enumerate(values):
            tokens[index, :len(item)] = item
            mask[index, :len(item)] = True
        return tokens, mask
    human_tokens, human_mask = pad(human)
    pseudo_tokens, pseudo_mask = pad(pseudo)
    return (
        torch.tensor(image_ids), torch.stack(images),
        human_tokens, human_mask, pseudo_tokens, pseudo_mask,
    )

train_set = SequenceKDDataset(train_groups, train_transform)
def make_train_loader(epoch):
    train_set.set_epoch(epoch)
    generator = torch.Generator().manual_seed(SEED + epoch)
    return DataLoader(
        train_set, batch_size=32, shuffle=True, generator=generator,
        num_workers=2, pin_memory=True, persistent_workers=False,
        collate_fn=sequence_collate,
    )

BASELINE_CIDER = float(baseline_result["best_validation_metrics"]["CIDEr_D"])
PSEUDO_SHA256 = sha256(PSEUDO_PARQUET_PATH)
BASELINE_SHA256 = sha256(BASELINE_BEST_PATH)
print("KD preflight           : PASSED")
print("Teacher / Student      :", TEACHER_NAME, "/", MODEL_NAME)
print("Human / pseudo pairs   :", len(train_set), "/", len(PSEUDO_BY_ID))
print("Loss                   : 0.5 CE_human + 0.5 CE_teacher_sequence")
print("Baseline CIDEr-D       :", BASELINE_CIDER)
print("Test                   : SEALED / NOT READ")

KD preflight           : PASSED
Teacher / Student      : BLIP Base / MobileViT-S
Human / pseudo pairs   : 29000 / 29000
Loss                   : 0.5 CE_human + 0.5 CE_teacher_sequence
Baseline CIDEr-D       : 0.4852843242350992
Test                   : SEALED / NOT READ


## Cell 4 — Beam-3 và metrics Validation

In [4]:
TOKENS = vocab["tokens"]
SPECIAL = {0, 1, 2, 3}

def ids_to_caption(ids):
    words = []
    for x in ids:
        x = int(x)
        if x == 2:
            break
        if x not in SPECIAL:
            words.append(TOKENS[x])
    text = " ".join(words)
    for p in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + p, p)
    return text.strip()

@torch.inference_mode()
def beam_decode(memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    for _ in range(30):
        active = [b for b in beams if not b[2]]
        if not active:
            break
        seq = torch.tensor([b[0] for b in active], device=DEVICE)
        mem = memory.expand(len(active), -1, -1)
        logits = decoder(seq, mem)[:, -1].float()
        logits[:, 0] = -torch.inf
        logits[:, 1] = -torch.inf
        logp = logits.log_softmax(-1)
        candidates = [b for b in beams if b[2]]
        for i, (tokens, score, _) in enumerate(active):
            values, indices = logp[i].topk(beam_size)
            for value, index in zip(values.tolist(), indices.tolist()):
                new = tokens + [index]
                candidates.append((new, score + value, index == 2))
        beams = sorted(candidates, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha), reverse=True)[:beam_size]
    return max(beams, key=lambda b: b[1] / (max(1, len(b[0]) - 1) ** alpha))[0]

@torch.inference_mode()
def validate():
    from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    encoder.eval()
    decoder.eval()
    hypotheses = {}
    ce_sum, token_count = 0.0, 0
    for image_ids, images, tokens, mask in val_loader:
        images, tokens, mask = images.to(DEVICE), tokens.to(DEVICE), mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            logits = decoder(tokens[:, :-1], memory, mask[:, :-1])
        targets = tokens[:, 1:]
        valid = mask[:, 1:]
        losses = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]),
                                 targets.reshape(-1), reduction="none").reshape_as(targets)
        ce_sum += float((losses * valid).sum())
        token_count += int(valid.sum())
        for j, image_id in enumerate(image_ids.tolist()):
            hypotheses[int(image_id)] = [ids_to_caption(beam_decode(memory[j:j+1]))]
    refs = {int(k): [{"caption": x} for x in v] for k, v in VAL_REFERENCES.items()}
    hyps = {int(k): [{"caption": v[0]}] for k, v in hypotheses.items()}
    tokenizer = PTBTokenizer()
    refs = tokenizer.tokenize(refs)
    hyps = tokenizer.tokenize(hyps)
    b, _ = Bleu(4).compute_score(refs, hyps)
    meteor = Meteor()
    try:
        m, _ = meteor.compute_score(refs, hyps)
    finally:
        close_method = getattr(meteor, "close", None)
        if callable(close_method):
            close_method()
    r, _ = Rouge().compute_score(refs, hyps)
    c, _ = Cider().compute_score(refs, hyps)
    return {"ce_loss": ce_sum / token_count, "BLEU_1": b[0], "BLEU_4": b[3],
            "METEOR": m, "ROUGE_L": r, "CIDEr_D": c}, hypotheses

## Cell 5 — Full Sequence-KD, Early Stopping và resume

In [5]:
MAX_EPOCHS = int(protocol["maximum_epochs"])
PATIENCE = int(protocol["early_stopping"]["patience"])
MIN_EPOCHS = int(protocol["early_stopping"]["minimum_completed_epochs"])
MIN_DELTA = float(protocol["minimum_cider_improvement"])
ACCUM = 2
ALPHA_HUMAN = 0.5
ALPHA_SEQUENCE = 0.5
steps_per_epoch = math.ceil(math.ceil(len(train_set) / 32) / ACCUM)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup = max(1, round(0.05 * total_steps))

optimizer = torch.optim.AdamW([
    {"params": encoder.backbone.parameters(), "lr": 1e-4},
    {"params": list(encoder.projection.parameters()) +
               [encoder.visual_position_embedding] +
               list(encoder.output_norm.parameters()), "lr": 3e-4},
    {"params": decoder.parameters(), "lr": 3e-4},
], betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4)

def lr_lambda(step):
    if step < warmup:
        return max(1e-8, step / warmup)
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", init_scale=1024.0)
LAST = WORK_ROOT / "last.pt"
BEST = WORK_ROOT / "best.pt"
METRICS_CSV = WORK_ROOT / "metrics.csv"
CAPTIONS = WORK_ROOT / "best_validation_captions.json"

def state_cpu(module):
    return {key: value.detach().cpu() for key, value in module.state_dict().items()}

def save_checkpoint(path, epoch, best, stale, global_step, metrics):
    payload = {
        "schema_version": "f30k_sequence_kd_confirmation_v1",
        "experiment_id": EXPERIMENT_ID, "model_id": MODEL_ID,
        "teacher_id": TEACHER_ID, "seed": SEED,
        "epoch": epoch, "best_cider": best, "stale_evaluations": stale,
        "global_step": global_step, "metrics": metrics,
        "encoder_state_dict": state_cpu(encoder),
        "decoder_state_dict": state_cpu(decoder),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all(),
        "baseline_checkpoint_sha256": BASELINE_SHA256,
        "pseudo_corpus_sha256": PSEUDO_SHA256,
        "protocol_sha256": sha256(PROTOCOL_PATH),
        "data_contract_sha256": sha256(DATA_CONTRACT_PATH),
        "test_set_status": "sealed", "test_set_used": False,
    }
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)

start_epoch, best_cider, stale, global_step = 0, -math.inf, 0, 0
history = []
resume_candidates = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}/last.pt"))
if resume_candidates:
    assert len(resume_candidates) == 1, resume_candidates
    payload = torch.load(resume_candidates[0], map_location="cpu", weights_only=False)
    assert payload["experiment_id"] == EXPERIMENT_ID
    assert payload["baseline_checkpoint_sha256"] == BASELINE_SHA256
    assert payload["pseudo_corpus_sha256"] == PSEUDO_SHA256
    assert payload["test_set_used"] is False
    encoder.load_state_dict(payload["encoder_state_dict"], strict=True)
    decoder.load_state_dict(payload["decoder_state_dict"], strict=True)
    optimizer.load_state_dict(payload["optimizer_state_dict"])
    scheduler.load_state_dict(payload["scheduler_state_dict"])
    scaler.load_state_dict(payload["scaler_state_dict"])
    random.setstate(payload["python_rng_state"])
    np.random.set_state(payload["numpy_rng_state"])
    torch.set_rng_state(payload["torch_rng_state"])
    torch.cuda.set_rng_state_all(payload["cuda_rng_state_all"])
    start_epoch = int(payload["epoch"]) + 1
    best_cider = float(payload["best_cider"])
    stale = int(payload["stale_evaluations"])
    global_step = int(payload["global_step"])
    resume_root = resume_candidates[0].parent
    for name, destination in [
        ("metrics.csv", METRICS_CSV), ("best.pt", BEST),
        ("best_validation_captions.json", CAPTIONS),
    ]:
        source = resume_root / name
        if source.is_file():
            shutil.copy2(source, destination)
    if METRICS_CSV.exists():
        history = pd.read_csv(METRICS_CSV).to_dict("records")
    print("Exact resume           :", resume_candidates[0], "next epoch", start_epoch + 1)

run_started = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    epoch_started = time.time()
    encoder.train(); decoder.train()
    for module in encoder.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
    loader = make_train_loader(epoch)
    optimizer.zero_grad(set_to_none=True)
    human_sum = pseudo_sum = 0.0
    human_count = pseudo_count = 0
    for micro_step, (_, images, human, human_mask, pseudo, pseudo_mask) in enumerate(loader):
        images = images.to(DEVICE)
        human, human_mask = human.to(DEVICE), human_mask.to(DEVICE)
        pseudo, pseudo_mask = pseudo.to(DEVICE), pseudo_mask.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            memory = encoder(images)
            human_logits = decoder(human[:, :-1], memory, human_mask[:, :-1])
            pseudo_logits = decoder(pseudo[:, :-1], memory, pseudo_mask[:, :-1])
            human_target, human_valid = human[:, 1:], human_mask[:, 1:]
            pseudo_target, pseudo_valid = pseudo[:, 1:], pseudo_mask[:, 1:]
            human_raw = F.cross_entropy(
                human_logits.float().reshape(-1, human_logits.shape[-1]),
                human_target.reshape(-1), reduction="none",
            ).reshape_as(human_target)
            pseudo_raw = F.cross_entropy(
                pseudo_logits.float().reshape(-1, pseudo_logits.shape[-1]),
                pseudo_target.reshape(-1), reduction="none",
            ).reshape_as(pseudo_target)
            human_ce = (human_raw * human_valid).sum() / human_valid.sum()
            pseudo_ce = (pseudo_raw * pseudo_valid).sum() / pseudo_valid.sum()
            loss = ALPHA_HUMAN * human_ce + ALPHA_SEQUENCE * pseudo_ce
        scaler.scale(loss / ACCUM).backward()
        human_sum += float((human_raw.detach() * human_valid).sum())
        human_count += int(human_valid.sum())
        pseudo_sum += float((pseudo_raw.detach() * pseudo_valid).sum())
        pseudo_count += int(pseudo_valid.sum())
        final_micro = micro_step + 1 == len(loader)
        if (micro_step + 1) % ACCUM == 0 or final_micro:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(decoder.parameters()), 1.0
            )
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
        if (micro_step + 1) % 100 == 0 or final_micro:
            print(
                f"epoch={epoch+1:02d} batch={micro_step+1:04d}/{len(loader):04d} "
                f"human_CE={human_sum/human_count:.6f} "
                f"pseudo_CE={pseudo_sum/pseudo_count:.6f}",
                flush=True,
            )
    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | VALIDATION START", flush=True)
    metrics, hypotheses = validate()
    metrics.update({
        "epoch": epoch + 1,
        "train_human_ce": human_sum / human_count,
        "train_teacher_sequence_ce": pseudo_sum / pseudo_count,
        "train_weighted_loss": (
            ALPHA_HUMAN * human_sum / human_count +
            ALPHA_SEQUENCE * pseudo_sum / pseudo_count
        ),
        "global_step": global_step,
        "elapsed_hours": (time.time() - run_started) / 3600,
    })
    history.append(metrics)
    pd.DataFrame(history).to_csv(METRICS_CSV, index=False)
    improved = metrics["CIDEr_D"] > best_cider + MIN_DELTA
    if improved:
        best_cider, stale = metrics["CIDEr_D"], 0
    else:
        stale += 1
    save_checkpoint(LAST, epoch, best_cider, stale, global_step, metrics)
    if improved:
        save_checkpoint(BEST, epoch, best_cider, stale, global_step, metrics)
        CAPTIONS.write_text(
            json.dumps(
                {"references": VAL_REFERENCES, "hypotheses": hypotheses},
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
    print(
        f"Epoch {epoch+1:02d} | CIDEr-D={metrics['CIDEr_D']:.6f} | "
        f"delta_baseline={metrics['CIDEr_D']-BASELINE_CIDER:+.6f} | "
        f"stale={stale}/{PATIENCE} | seconds={time.time()-epoch_started:.1f}",
        flush=True,
    )
    if epoch + 1 >= MIN_EPOCHS and stale >= PATIENCE:
        print("Early stopping         : TRIGGERED")
        break

assert BEST.exists() and LAST.exists()
best_payload = torch.load(BEST, map_location="cpu", weights_only=False)
best_metrics = best_payload["metrics"]
summary = {
    "schema_version": "f30k_sequence_kd_result_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "complete", "experiment_id": EXPERIMENT_ID,
    "teacher_id": TEACHER_ID, "teacher_name": TEACHER_NAME,
    "student_id": MODEL_ID, "student_name": MODEL_NAME, "seed": SEED,
    "method": "Sequence KD",
    "loss": "0.5 CE_human + 0.5 CE_teacher_sequence",
    "best_epoch": int(best_payload["epoch"]) + 1,
    "best_validation_metrics": best_metrics,
    "baseline_validation_cider_d": BASELINE_CIDER,
    "delta_validation_cider_d": float(best_metrics["CIDEr_D"]) - BASELINE_CIDER,
    "selection_metric": "Validation CIDEr-D",
    "maximum_epochs": MAX_EPOCHS, "early_stopping_enabled": True,
    "next_phase_authorized": "F30K_T2S2_SeqKD_Aggregate",
    "test_set_status": "sealed", "test_set_used": False,
    "artifacts": {},
}
for path in [BEST, LAST, METRICS_CSV, CAPTIONS]:
    summary["artifacts"][path.name] = {
        "bytes": path.stat().st_size, "sha256": sha256(path)
    }
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
result_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
manifest = {
    "schema_version": "f30k_kd_manifest_v1",
    "experiment_id": EXPERIMENT_ID, "status": "complete",
    "artifacts": [
        {
            "relative_path": path.name, "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in [result_path, BEST, LAST, METRICS_CSV, CAPTIONS]
    ],
    "test_set_status": "sealed", "test_set_used": False,
}
(WORK_ROOT / "MANIFEST.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Status                 : COMPLETE")
print("Best Validation        :", best_metrics)
print("Delta vs baseline      :", summary["delta_validation_cider_d"])
print("Next                   :", summary["next_phase_authorized"])
print("Held-out Test          : SEALED / NOT USED")

epoch=01 batch=0100/0907 human_CE=2.564240 pseudo_CE=1.558776
epoch=01 batch=0200/0907 human_CE=2.573934 pseudo_CE=1.489965
epoch=01 batch=0300/0907 human_CE=2.595416 pseudo_CE=1.442961
epoch=01 batch=0400/0907 human_CE=2.600019 pseudo_CE=1.410409
epoch=01 batch=0500/0907 human_CE=2.612006 pseudo_CE=1.386241
epoch=01 batch=0600/0907 human_CE=2.622758 pseudo_CE=1.365973
epoch=01 batch=0700/0907 human_CE=2.622893 pseudo_CE=1.347415
epoch=01 batch=0800/0907 human_CE=2.633585 pseudo_CE=1.334064
epoch=01 batch=0900/0907 human_CE=2.640897 pseudo_CE=1.322852
epoch=01 batch=0907/0907 human_CE=2.641407 pseudo_CE=1.321784
Epoch 01/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 277058.97 tokens per second.
PTBTokenizer tokenized 11579 tokens at 101884.10 tokens per second.


{'testlen': 9552, 'reflen': 9698, 'guess': [9552, 8538, 7524, 6510], 'correct': [5798, 2424, 978, 392]}
ratio: 0.984945349556508
Epoch 01 | CIDEr-D=0.372543 | delta_baseline=-0.112741 | stale=0/4 | seconds=2159.0
epoch=02 batch=0100/0907 human_CE=2.710610 pseudo_CE=1.186648
epoch=02 batch=0200/0907 human_CE=2.701618 pseudo_CE=1.178494
epoch=02 batch=0300/0907 human_CE=2.708346 pseudo_CE=1.175138
epoch=02 batch=0400/0907 human_CE=2.706932 pseudo_CE=1.172256
epoch=02 batch=0500/0907 human_CE=2.712388 pseudo_CE=1.168972
epoch=02 batch=0600/0907 human_CE=2.710309 pseudo_CE=1.168119
epoch=02 batch=0700/0907 human_CE=2.709787 pseudo_CE=1.163485
epoch=02 batch=0800/0907 human_CE=2.716775 pseudo_CE=1.159494
epoch=02 batch=0900/0907 human_CE=2.716684 pseudo_CE=1.155941
epoch=02 batch=0907/0907 human_CE=2.717434 pseudo_CE=1.155551
Epoch 02/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 297146.07 tokens per second.
PTBTokenizer tokenized 11554 tokens at 94679.06 tokens per second.


{'testlen': 9547, 'reflen': 9693, 'guess': [9547, 8533, 7519, 6505], 'correct': [5829, 2409, 954, 393]}
ratio: 0.984937583823276
Epoch 02 | CIDEr-D=0.366900 | delta_baseline=-0.118385 | stale=1/4 | seconds=254.3
epoch=03 batch=0100/0907 human_CE=2.691806 pseudo_CE=1.079105
epoch=03 batch=0200/0907 human_CE=2.682906 pseudo_CE=1.075420
epoch=03 batch=0300/0907 human_CE=2.682151 pseudo_CE=1.074415
epoch=03 batch=0400/0907 human_CE=2.686196 pseudo_CE=1.073645
epoch=03 batch=0500/0907 human_CE=2.685976 pseudo_CE=1.072193
epoch=03 batch=0600/0907 human_CE=2.685619 pseudo_CE=1.071858
epoch=03 batch=0700/0907 human_CE=2.686862 pseudo_CE=1.071082
epoch=03 batch=0800/0907 human_CE=2.688010 pseudo_CE=1.070032
epoch=03 batch=0900/0907 human_CE=2.686363 pseudo_CE=1.067790
epoch=03 batch=0907/0907 human_CE=2.686357 pseudo_CE=1.067205
Epoch 03/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 336885.02 tokens per second.
PTBTokenizer tokenized 11461 tokens at 105639.14 tokens per second.


... (31 outputs left out)

PTBTokenizer tokenized 74030 tokens at 322329.39 tokens per second.
PTBTokenizer tokenized 11762 tokens at 98791.34 tokens per second.


{'testlen': 9793, 'reflen': 9864, 'guess': [9793, 8779, 7765, 6751], 'correct': [6118, 2615, 1067, 418]}
ratio: 0.9928021086779204
Epoch 19 | CIDEr-D=0.424997 | delta_baseline=-0.060288 | stale=3/4 | seconds=258.6
epoch=20 batch=0100/0907 human_CE=2.419420 pseudo_CE=0.623986
epoch=20 batch=0200/0907 human_CE=2.412710 pseudo_CE=0.626914
epoch=20 batch=0300/0907 human_CE=2.411239 pseudo_CE=0.625129
epoch=20 batch=0400/0907 human_CE=2.407896 pseudo_CE=0.624447
epoch=20 batch=0500/0907 human_CE=2.408846 pseudo_CE=0.624719
epoch=20 batch=0600/0907 human_CE=2.405938 pseudo_CE=0.625774
epoch=20 batch=0700/0907 human_CE=2.408971 pseudo_CE=0.625038
epoch=20 batch=0800/0907 human_CE=2.406722 pseudo_CE=0.625160
epoch=20 batch=0900/0907 human_CE=2.406170 pseudo_CE=0.625110
epoch=20 batch=0907/0907 human_CE=2.407340 pseudo_CE=0.624960
Epoch 20/20 | VALIDATION START


PTBTokenizer tokenized 74030 tokens at 312602.76 tokens per second.
PTBTokenizer tokenized 11786 tokens at 101080.74 tokens per second.


{'testlen': 9818, 'reflen': 9846, 'guess': [9818, 8804, 7790, 6776], 'correct': [6151, 2648, 1090, 428]}
ratio: 0.9971562055656107
Epoch 20 | CIDEr-D=0.429594 | delta_baseline=-0.055690 | stale=4/4 | seconds=258.7
Early stopping         : TRIGGERED
Status                 : COMPLETE
Best Validation        : {'ce_loss': 2.895193969632227, 'BLEU_1': 0.625916976580249, 'BLEU_4': 0.20045233008908564, 'METEOR': 0.18467107899720714, 'ROUGE_L': np.float64(0.43107662208435427), 'CIDEr_D': np.float64(0.4374363800976886), 'epoch': 16, 'train_human_ce': 2.4148098765442425, 'train_teacher_sequence_ce': 0.6527249940218988, 'train_weighted_loss': 1.5337674352830706, 'global_step': 7264, 'elapsed_hours': 1.675102947751681}
Delta vs baseline      : -0.047847944137410614
Next                   : F30K_T2S2_SeqKD_Aggregate
Held-out Test          : SEALED / NOT USED
